In [1]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent.parent)

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import cv2
import time

from ml.src.inference.video_pipeline import VideoPipeline
from backend.app.core.config import get_settings

In [5]:
settings = get_settings()
pipeline = VideoPipeline(settings)

COLORS = {
    "alert": (0, 255, 0),          # Зеленый
    "mild_fatigue": (0, 255, 255), # Желтый
    "severe_fatigue": (0, 0, 255)  # Красный
}

VIDEO_SOURCE = 0 # "10.mov"
is_live_stream = isinstance(VIDEO_SOURCE, int)
print(is_live_stream)

cap = cv2.VideoCapture(VIDEO_SOURCE)

video_fps = cap.get(cv2.CAP_PROP_FPS)
if video_fps <= 0:
    video_fps = 30.0
print(f"Video FPS: {video_fps}")

frame_index = 0

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Не удалось получить кадр с камеры.")
            break
            
        if is_live_stream:
            frame = cv2.flip(frame, 1)

        start_time = time.perf_counter()

        if not is_live_stream:
            current_time = frame_index / video_fps
        else:
            current_time = time.time()
        
        frame_index += 1
        
        result = pipeline.process_frame(frame, timestamp=current_time)

        elapsed = time.perf_counter() - start_time
        fps = 1.0 / elapsed if elapsed > 0 else 0.0
        
        if result and result.fatigue_score:
            score = result.fatigue_score
            color = COLORS.get(score.level, (255, 255, 255))
            
            cv2.putText(frame, f"STATUS: {score.level.upper()}", (10, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
            cv2.putText(frame, f"Conf: {score.confidence:.2f}", (10, 70), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            metrics_y = 110
            cv2.putText(frame, f"PERCLOS: {score.perclos*100:.1f}%", (10, metrics_y), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)
            cv2.putText(frame, f"Blinks/min: {score.blink_rate:.1f}", (10, metrics_y + 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)
            cv2.putText(frame, f"Yawns (5m): {score.yawn_count}", (10, metrics_y + 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)
            
            if score.head_down:
                cv2.putText(frame, "WARNING: HEAD DOWN!", (10, metrics_y + 90), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                            
            if score.face_absent:
                cv2.putText(frame, "WARNING: FACE NOT DETECTED!", (10, metrics_y + 120), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            if result.landmarks:
                for pt in result.landmarks.points:
                    x = int(pt[0] * frame.shape[1])
                    y = int(pt[1] * frame.shape[0])
                    cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

        ear_history = pipeline._window.ear_history

        if ear_history:
            last_ear = ear_history[-1]
            window_size = len(ear_history)
            ear_color = (0, 0, 255) if last_ear < settings.EAR_THRESHOLD else (0, 255, 0)

            cv2.putText(frame,
                        f"EAR: {last_ear:.3f}  thresh: {settings.EAR_THRESHOLD:.2f}",
                        (10, frame.shape[1] - 140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, ear_color, 2)

            cv2.putText(frame,
                        f"Window: {window_size} frames",
                        (10, frame.shape[1] - 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (150, 150, 150), 1)

            graph_x, graph_y = 10, frame.shape[0] - 20
            graph_h, graph_w = 40, 300
            history = ear_history[-100:]

            cv2.rectangle(frame,
                          (graph_x, graph_y - graph_h),
                          (graph_x + graph_w, graph_y),
                          (30, 30, 30), -1)

            threshold_y = int(graph_y - (settings.EAR_THRESHOLD / 0.5) * graph_h)
            cv2.line(frame, (graph_x, threshold_y),
                     (graph_x + graph_w, threshold_y), (0, 0, 255), 1)

            for i in range(1, len(history)):
                x1 = graph_x + int((i - 1) / max(len(history), 1) * graph_w)
                x2 = graph_x + int(i / max(len(history), 1) * graph_w)
                y1 = int(graph_y - (min(history[i-1], 0.5) / 0.5) * graph_h)
                y2 = int(graph_y - (min(history[i], 0.5) / 0.5) * graph_h)
                cv2.line(frame, (x1, y1), (x2, y2), (0, 200, 255), 1)

            cv2.putText(frame, "EAR (last 100 frames)",
                        (graph_x, graph_y - graph_h - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)

        else:
            cv2.putText(frame, "EAR: no face detected",
                        (10, frame.shape[1] - 140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

        if hasattr(pipeline, '_window') and pipeline._window.head_poses:
            last_pose = pipeline._window.head_poses[-1]
            if last_pose is not None:
                pitch_color = (0, 0, 255) if last_pose.pitch > settings.HEAD_PITCH_THRESHOLD_DEG else (0, 255, 0)
                cv2.putText(frame,
                            f"Pitch: {last_pose.pitch:.1f} Yaw: {last_pose.yaw:.1f} Roll: {last_pose.roll:.1f}",
                            (frame.shape[1] // 2 - 150, frame.shape[0] - 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, pitch_color, 2)
                cv2.putText(frame,
                            f"HEAD_DOWN thresh: {settings.HEAD_PITCH_THRESHOLD_DEG:.1f}°",
                            (frame.shape[1] // 2 - 150, frame.shape[0] - 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

        cv2.putText(frame, f"FPS: {fps:.1f}", (frame.shape[1] - 120, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        cv2.imshow("Fatigue Detection System", frame)

        wait_time = 1 if is_live_stream else 30
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Прервано пользователем.")
            break

except Exception as e:
    print(f"Произошла ошибка: {e}")

finally:
    cap.release()
    cv2.destroyAllWindows()
    pipeline.reset_session()
    for i in range(5):
        cv2.waitKey(1)
    print("Ресурсы освобождены.")

True
Video FPS: 30.0
Прервано пользователем.
Ресурсы освобождены.
